# Tutorial 5: Probability and Statistics — Distributions, CLT, and Law of Large Numbers

## Context

In **Lectures 6-7**, you learned about probability distributions — the mathematical models that describe how random variables behave. In this tutorial, you'll see these distributions *in action* using real-world computing scenarios.

### The Big Questions

- **How long will a server take to respond?** (Continuous distributions)
- **How many errors will we see per hour?** (Discrete distributions)
- **Why do averages behave more predictably than individual observations?** (Law of Large Numbers & Central Limit Theorem)

Understanding distributions helps us:
- **Set realistic SLA thresholds** (Service Level Agreements)
- **Predict system failures** before they happen
- **Design better load tests** and monitoring
- **Build confidence intervals** for system metrics

Let's model real-world computing systems: tracking server response times, error counts, session durations, and throughput.

## Setup: Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set style for better-looking plots
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Seed for reproducibility
rng = np.random.default_rng(42)

print("Libraries imported successfully!")


---

# Section 1: Discrete Distributions — PMFs and CDFs

**Lecture Connection:** In Lecture 6, you computed probability mass functions (PMFs) and cumulative distribution functions (CDFs) by hand. Now we'll use `scipy.stats` to do the heavy lifting.

### Why Discrete Distributions?

Discrete distributions model *countable* outcomes:
- Does a request succeed or fail? (Bernoulli)
- Out of n requests, how many fail? (Binomial)
- How many errors occur in an hour? (Poisson)

**Resources:**
- [scipy.stats documentation](https://docs.scipy.org/doc/scipy/reference/stats.html)
- [Bernoulli & Binomial](https://en.wikipedia.org/wiki/Bernoulli_distribution)
- [Poisson Distribution](https://en.wikipedia.org/wiki/Poisson_distribution)

## 1.1: Bernoulli Distribution — Single Request Success/Failure

**Scenario:** Your server successfully handles a request 99% of the time. A single request is a Bernoulli trial with p=0.99.

In [ ]:
# Bernoulli distribution: single trial, probability of success p
p_success = 0.99  # 99% success rate

bernoulli_dist = stats.binom(n=1, p=p_success)

# PMF: P(X = k) for k in {0, 1}
k_values = [0, 1]
pmf = [bernoulli_dist.pmf(k) for k in k_values]

print("Bernoulli Distribution (p = 0.99)")
print(f"P(Request fails, X=0) = {pmf[0]:.4f}")
print(f"P(Request succeeds, X=1) = {pmf[1]:.4f}")

# CDF: P(X <= k)
print(f"\nCDF: P(X <= 0) = {bernoulli_dist.cdf(0):.4f}")
print(f"CDF: P(X <= 1) = {bernoulli_dist.cdf(1):.4f}")


## 1.2: Binomial Distribution — Multiple Requests

**Scenario:** Your server handles 10 requests in a short time window. What's the probability exactly 9 succeed? What's P(at least 8 succeed)?

A Binomial(n, p) distribution counts successes in n independent Bernoulli trials.

In [ ]:
# Binomial distribution: n=10 requests, p=0.99 success rate
n_requests = 10
p_success = 0.99

binom_dist = stats.binom(n=n_requests, p=p_success)

# PMF: probability of exactly k successes
k = 9
pmf_9 = binom_dist.pmf(k)
print(f"P(exactly 9 successes out of 10) = {pmf_9:.4f}")

# CDF: P(X <= k)
cdf_8 = binom_dist.cdf(8)
prob_at_least_9 = 1 - cdf_8  # P(X >= 9)
print(f"P(at least 9 successes) = {prob_at_least_9:.4f}")

# Expected value and standard deviation
mean = binom_dist.mean()
std = binom_dist.std()
print(f"\nExpected value (mean): {mean:.2f}")
print(f"Standard deviation: {std:.4f}")


### Visualizing the Binomial PMF

Let's plot the PMF to see the probability of different numbers of successes.

In [ ]:
# Generate PMF for all possible values
x_values = np.arange(0, n_requests + 1)
pmf_values = binom_dist.pmf(x_values)

fig, ax = plt.subplots(1, 1, figsize=(10, 6))
ax.bar(x_values, pmf_values, color='steelblue', alpha=0.7, edgecolor='black')
ax.set_xlabel('Number of Successful Requests (out of 10)', fontsize=12)
ax.set_ylabel('Probability (PMF)', fontsize=12)
ax.set_title('Binomial Distribution: 10 Requests, 99% Success Rate', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("This bar chart shows the PMF — notice it's concentrated near n=10 (high success rate).")


## 1.3: Poisson Distribution — Rare Events Over Time

**Scenario:** Your server logs show 2 errors per hour on average. What's the probability of seeing more than 3 errors in a given hour?

The **Poisson distribution** models the number of events in a fixed time interval when events occur independently at a constant average rate (λ = lambda).

**When to use Poisson vs Binomial:**
- **Binomial:** Fixed number of trials (n), asking about successes
- **Poisson:** Events occurring over time/space, n unknown, asking about count in interval

**Resources:**
- [Poisson approximation to Binomial](https://en.wikipedia.org/wiki/Poisson_distribution#Poisson_as_a_limit_of_binomial_distributions)

In [ ]:
# Poisson distribution: lambda = 2 errors/hour
lambda_errors = 2.0

poisson_dist = stats.poisson(mu=lambda_errors)

# PMF: probability of exactly k errors
k = 3
pmf_3 = poisson_dist.pmf(k)
print(f"P(exactly 3 errors in an hour) = {pmf_3:.4f}")

# CDF: P(X <= 3)
cdf_3 = poisson_dist.cdf(3)
prob_more_than_3 = 1 - cdf_3  # P(X > 3)
print(f"P(more than 3 errors in an hour) = {prob_more_than_3:.4f}")

# Expected value and variance (equal for Poisson!)
mean = poisson_dist.mean()
variance = poisson_dist.var()
print(f"\nExpected value: {mean:.2f}")
print(f"Variance: {variance:.2f}")
print("(Note: For Poisson, mean = variance = λ)")


### Visualizing Poisson PMF and CDF

In [ ]:
# Generate PMF and CDF for Poisson
x_values = np.arange(0, 12)
pmf_values = poisson_dist.pmf(x_values)
cdf_values = poisson_dist.cdf(x_values)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PMF
ax1.bar(x_values, pmf_values, color='coral', alpha=0.7, edgecolor='black')
ax1.set_xlabel('Number of Errors per Hour', fontsize=12)
ax1.set_ylabel('Probability (PMF)', fontsize=12)
ax1.set_title('Poisson Distribution (λ=2)', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# CDF
ax2.step(x_values, cdf_values, where='mid', color='green', linewidth=2, label='CDF')
ax2.scatter(x_values, cdf_values, color='green', s=50, zorder=5)
ax2.set_xlabel('Number of Errors per Hour', fontsize=12)
ax2.set_ylabel('Cumulative Probability (CDF)', fontsize=12)
ax2.set_title('Poisson CDF (λ=2)', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0, 1.05])

plt.tight_layout()
plt.show()


## 1.4: Case Study — Modeling System Errors

**Real-world scenario:**
Your server handles **1000 requests per hour** with a **0.1% error rate**. You want to know:
1. What distribution models the number of errors per hour?
2. What's the probability of seeing more than 3 errors in an hour?
3. How often would we expect to see 0 errors?

In [ ]:
# Setup
n_requests_per_hour = 1000
error_rate = 0.001  # 0.1%
lambda_errors = n_requests_per_hour * error_rate  # Expected errors per hour

print(f"Number of requests per hour: {n_requests_per_hour}")
print(f"Error rate per request: {error_rate:.4f} ({error_rate*100:.2f}%)")
print(f"Expected errors per hour (λ): {lambda_errors:.1f}")
print()

# Model 1: Binomial (exact)
binom_errors = stats.binom(n=n_requests_per_hour, p=error_rate)
prob_more_than_3_binom = 1 - binom_errors.cdf(3)
prob_zero_binom = binom_errors.pmf(0)

print("Using Binomial(n=1000, p=0.001):")
print(f"  P(more than 3 errors) = {prob_more_than_3_binom:.6f}")
print(f"  P(0 errors) = {prob_zero_binom:.6f}")
print()

# Model 2: Poisson (approximation, simpler)
poisson_errors = stats.poisson(mu=lambda_errors)
prob_more_than_3_poisson = 1 - poisson_errors.cdf(3)
prob_zero_poisson = poisson_errors.pmf(0)

print(f"Using Poisson(λ={lambda_errors}):")
print(f"  P(more than 3 errors) = {prob_more_than_3_poisson:.6f}")
print(f"  P(0 errors) = {prob_zero_poisson:.6f}")
print()

print("Note: Binomial and Poisson give similar results (Poisson approximation valid when n is large, p is small).")


## 1.5: Exercise 1 — Discrete Distributions

In [ ]:
# EXERCISE 1: System Reliability
# Your database connection has a 99.5% reliability rate.
# Over 50 independent operations:
#   (a) What's the probability all 50 succeed?
#   (b) What's the probability at least 48 succeed?
#   (c) What's the expected number of failures?

n_ops = 50
p_success = 0.995
p_failure = 1 - p_success

reliability_dist = stats.binom(n=n_ops, p=p_success)

prob_all_50 = reliability_dist.pmf(50)
prob_at_least_48 = 1 - reliability_dist.cdf(47)
expected_failures = n_ops * p_failure

print(f"Binomial model: X ~ Binomial(n={n_ops}, p={p_success}) where X = # successful operations")
print(f"(a) P(all 50 succeed) = {prob_all_50:.6f}")
print(f"(b) P(at least 48 succeed) = {prob_at_least_48:.6f}")
print(f"(c) Expected number of failures = {expected_failures:.3f}")

## 1.6: Exercise 2 — Poisson Rare Events

In [ ]:
# EXERCISE 2: Server Crashes
# Your log data shows an average of 0.5 server crashes per month.
#   (a) What's P(exactly 1 crash next month)?
#   (b) What's P(0 crashes next month)?
#   (c) What's P(more than 2 crashes next month)?
# Use Poisson(λ=0.5).

lambda_crashes = 0.5
crash_dist = stats.poisson(mu=lambda_crashes)

prob_exactly_1 = crash_dist.pmf(1)
prob_zero = crash_dist.pmf(0)
prob_more_than_2 = 1 - crash_dist.cdf(2)

print(f"Poisson model: X ~ Poisson(λ={lambda_crashes})")
print(f"(a) P(exactly 1 crash) = {prob_exactly_1:.6f}")
print(f"(b) P(0 crashes) = {prob_zero:.6f}")
print(f"(c) P(more than 2 crashes) = {prob_more_than_2:.6f}")

## 1.7: Exercise 3 — Comparing Distributions

In [ ]:
# EXERCISE 3: Binomial vs Poisson
# Generate the PMF for both Binomial(n=100, p=0.05) and Poisson(λ=5).
# Plot them side-by-side on the same figure.
# Comment: At what values of n and p is the Poisson approximation reasonable?

n = 100
p = 0.05
lambda_poisson = n * p

binom_compare = stats.binom(n=n, p=p)
poisson_compare = stats.poisson(mu=lambda_poisson)

x = np.arange(0, 16)
binom_pmf = binom_compare.pmf(x)
poisson_pmf = poisson_compare.pmf(x)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x, binom_pmf, 'o-', linewidth=2, label=f'Binomial(n={n}, p={p})')
ax.plot(x, poisson_pmf, 's--', linewidth=2, label=f'Poisson(λ={lambda_poisson})')
ax.set_xlabel('Number of Events', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('Binomial vs Poisson PMF', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

max_abs_diff = np.max(np.abs(binom_pmf - poisson_pmf))
print(f"Maximum absolute difference over k=0,...,15: {max_abs_diff:.4f}")
print("Poisson is a reasonable approximation when n is large and p is small,")
print("so that λ = np stays moderate. A common rule of thumb is n >= 20 and p <= 0.05.")

---

# Section 2: Continuous Distributions — PDFs and CDFs

**Lecture Connection:** In Lecture 6, you worked with probability density functions (PDFs) and cumulative distribution functions (CDFs). Now we'll compute probabilities and visualize regions for continuous random variables.

### Why Continuous Distributions?

Continuous distributions model *uncountable* outcomes:
- How long does a server request take? (Normal, Exponential, Lognormal)
- What's the time between requests? (Exponential)
- How long is a user session? (Exponential, Weibull)

**Key difference from discrete:**
- For continuous: P(X = x) = 0 (no single point has positive probability)
- Instead, we compute P(a < X < b) (areas under the PDF)

**Resources:**
- [Normal Distribution](https://en.wikipedia.org/wiki/Normal_distribution)
- [Exponential Distribution](https://en.wikipedia.org/wiki/Exponential_distribution)

## 2.1: Normal Distribution — Symmetric, Bell-Shaped

**Scenario:** User session durations follow a normal distribution with mean 30 minutes and standard deviation 5 minutes.

In [ ]:
# Normal distribution: mean=30, std=5
mu = 30  # mean session duration (minutes)
sigma = 5  # standard deviation (minutes)

normal_dist = stats.norm(loc=mu, scale=sigma)

# PDF: probability density at a point
x_val = 30
pdf_at_30 = normal_dist.pdf(x_val)
print(f"PDF at x={x_val}: {pdf_at_30:.6f}")

# CDF: P(X <= x)
cdf_at_30 = normal_dist.cdf(x_val)
print(f"CDF: P(X <= 30) = {cdf_at_30:.4f}")

# Probability in a range: P(25 < X < 35)
prob_25_35 = normal_dist.cdf(35) - normal_dist.cdf(25)
print(f"P(25 < X < 35) = {prob_25_35:.4f}")

# Quantile function (inverse CDF): What value for the r.v. has cumulative probability p?
# Find x such that P(X <= x) = 0.95
x_95 = normal_dist.ppf(0.95)
print(f"\n95th percentile (ppf(0.95)): {x_95:.2f} minutes")


### The 68-95-99.7 Rule (Empirical Rule)

For a normal distribution:
- ~68% of data within 1σ of μ
- ~95% of data within 2σ of μ
- ~99.7% of data within 3σ of μ

Let's verify this computationally:

In [ ]:
# Verify the empirical rule
prob_1sigma = normal_dist.cdf(mu + sigma) - normal_dist.cdf(mu - sigma)
prob_2sigma = normal_dist.cdf(mu + 2*sigma) - normal_dist.cdf(mu - 2*sigma)
prob_3sigma = normal_dist.cdf(mu + 3*sigma) - normal_dist.cdf(mu - 3*sigma)

print("Empirical Rule Verification (μ=30, σ=5):")
print(f"P(μ - σ < X < μ + σ) = P(25 < X < 35) = {prob_1sigma:.4f} (~68%)")
print(f"P(μ - 2σ < X < μ + 2σ) = P(20 < X < 40) = {prob_2sigma:.4f} (~95%)")
print(f"P(μ - 3σ < X < μ + 3σ) = P(15 < X < 45) = {prob_3sigma:.4f} (~99.7%)")


### Visualizing the Normal Distribution

In [ ]:
# Plot PDF with shaded regions
x = np.linspace(mu - 4*sigma, mu + 4*sigma, 200)
pdf = normal_dist.pdf(x)

fig, ax = plt.subplots(figsize=(12, 6))

# PDF curve
ax.plot(x, pdf, 'b-', linewidth=2, label='PDF')

# Shade the region 25 < X < 35 (1σ)
x_fill = x[(x >= 25) & (x <= 35)]
ax.fill_between(x_fill, normal_dist.pdf(x_fill), alpha=0.4, color='green', label='P(25 < X < 35)')

# Shade the region X > 35
x_fill_right = x[x > 35]
ax.fill_between(x_fill_right, normal_dist.pdf(x_fill_right), alpha=0.3, color='red', label='P(X > 35)')

ax.axvline(mu, color='black', linestyle='--', linewidth=1, alpha=0.5, label=f'μ={mu}')
ax.set_xlabel('Session Duration (minutes)', fontsize=12)
ax.set_ylabel('Probability Density', fontsize=12)
ax.set_title('Normal Distribution: Session Durations (μ=30, σ=5)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 2.2: Exponential Distribution — Waiting Times

**Scenario:** The time between server requests follows an exponential distribution. On average, we see 1 request every 2 seconds, so λ = 0.5 requests/second (or mean waiting time = 2 seconds).

The **Exponential distribution** is memoryless: P(X > s+t | X > s) = P(X > t).

In [ ]:
# Exponential distribution: scale = 2 (mean waiting time in seconds)
# scale parameter = 1/λ where λ is the rate
mean_wait = 2  # seconds
expon_dist = stats.expon(scale=mean_wait)

# PDF at t=2
pdf_at_2 = expon_dist.pdf(2)
print(f"PDF at t=2 seconds: {pdf_at_2:.6f}")

# CDF: P(X <= t)
cdf_at_2 = expon_dist.cdf(2)
print(f"CDF: P(wait <= 2 seconds) = {cdf_at_2:.4f}")

# Probability wait > 3 seconds
prob_gt_3 = 1 - expon_dist.cdf(3)
print(f"P(wait > 3 seconds) = {prob_gt_3:.4f}")

# Mean and variance
print(f"\nMean waiting time: {expon_dist.mean():.2f} seconds")
print(f"Variance: {expon_dist.var():.2f} seconds²")


### The Memoryless Property

A key feature of the exponential distribution: the probability of waiting an additional time t doesn't depend on how long we've already waited.

In [ ]:
# Memoryless property: P(X > s+t | X > s) = P(X > t)
s = 2  # already waited 2 seconds
t = 1  # want to wait 1 more second

# P(X > 3 | X > 2) — given we've waited 2 sec, probability of waiting 3 more?
prob_gt_3_given_gt_2 = prob_gt_3 / (1 - expon_dist.cdf(s))

# P(X > 1) — fresh start, probability of waiting > 1 sec
prob_gt_1 = 1 - expon_dist.cdf(1)

print("Memoryless Property Verification:")
print(f"P(X > 3 | X > 2) = {prob_gt_3_given_gt_2:.4f}")
print(f"P(X > 1) = {prob_gt_1:.4f}")
print(f"Are they equal? {np.isclose(prob_gt_3_given_gt_2, prob_gt_1)}")


### Visualizing the Exponential Distribution

In [ ]:
# Plot exponential PDF and CDF
t = np.linspace(0, 10, 200)
pdf_exp = expon_dist.pdf(t)
cdf_exp = expon_dist.cdf(t)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# PDF
ax1.plot(t, pdf_exp, 'b-', linewidth=2)
ax1.fill_between(t[t > 3], pdf_exp[t > 3], alpha=0.4, color='red', label='P(X > 3)')
ax1.axvline(mean_wait, color='black', linestyle='--', linewidth=1, alpha=0.5, label=f'μ={mean_wait}')
ax1.set_xlabel('Waiting Time (seconds)', fontsize=12)
ax1.set_ylabel('Probability Density', fontsize=12)
ax1.set_title('Exponential PDF (λ=0.5, mean=2)', fontsize=13, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# CDF
ax2.plot(t, cdf_exp, 'g-', linewidth=2, label='CDF')
ax2.axhline(cdf_at_2, color='red', linestyle=':', alpha=0.5, label=f'P(X ≤ 2) = {cdf_at_2:.2f}')
ax2.axvline(2, color='red', linestyle=':', alpha=0.5)
ax2.set_xlabel('Waiting Time (seconds)', fontsize=12)
ax2.set_ylabel('Cumulative Probability', fontsize=12)
ax2.set_title('Exponential CDF', fontsize=13, fontweight='bold')
ax2.set_ylim([0, 1.05])
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 2.4: Exercise 4 — Continuous Distributions

In [ ]:
# EXERCISE 4: Disk Latency
# Disk access times follow a normal distribution with μ=5ms, σ=0.5ms.
#   (a) What's P(access time > 6ms)?
#   (b) What's the 99th percentile of access times?
#   (c) In a 1000-operation batch, how many would you expect to exceed 6ms?

mu_latency = 5
sigma_latency = 0.5

latency_dist = stats.norm(loc=mu_latency, scale=sigma_latency)

prob_gt_6 = 1 - latency_dist.cdf(6)
p99_latency = latency_dist.ppf(0.99)
expected_exceed_6 = 1000 * prob_gt_6

print(f"Normal model: X ~ Normal(μ={mu_latency} ms, σ={sigma_latency} ms)")
print(f"(a) P(access time > 6 ms) = {prob_gt_6:.6f}")
print(f"(b) 99th percentile = {p99_latency:.3f} ms")
print(f"(c) Expected number above 6 ms in 1000 operations = {expected_exceed_6:.2f}")

## 2.5: Exercise 5 — Normal Distribution Properties

In [ ]:
# EXERCISE 5: Verify the Empirical Rule
# For a Normal(μ=100, σ=15) distribution, compute:
#   (a) P(85 < X < 115)  — should be ~68%
#   (b) P(70 < X < 130)  — should be ~95%
#   (c) P(55 < X < 145)  — should be ~99.7%

mu = 100
sigma = 15
empirical_dist = stats.norm(loc=mu, scale=sigma)

prob_1sigma = empirical_dist.cdf(mu + sigma) - empirical_dist.cdf(mu - sigma)
prob_2sigma = empirical_dist.cdf(mu + 2*sigma) - empirical_dist.cdf(mu - 2*sigma)
prob_3sigma = empirical_dist.cdf(mu + 3*sigma) - empirical_dist.cdf(mu - 3*sigma)

print(f"(a) P(85 < X < 115) = {prob_1sigma:.4f} ({prob_1sigma*100:.2f}%)")
print(f"(b) P(70 < X < 130) = {prob_2sigma:.4f} ({prob_2sigma*100:.2f}%)")
print(f"(c) P(55 < X < 145) = {prob_3sigma:.4f} ({prob_3sigma*100:.2f}%)")
print("These match the 68-95-99.7 rule very closely.")

## 2.6: Exercise 6 — Exponential Waiting Times

In [ ]:
# EXERCISE 6: Queue Wait Times
# Request queue wait times follow Exponential with mean=10 seconds.
#   (a) What's P(wait <= 5 seconds)?
#   (b) What's P(wait > 20 seconds)?
#   (c) Find the 90th percentile of wait times.
#   (d) Plot the PDF and CDF.

mean_wait = 10
queue_dist = stats.expon(scale=mean_wait)

prob_wait_le_5 = queue_dist.cdf(5)
prob_wait_gt_20 = 1 - queue_dist.cdf(20)
p90_wait = queue_dist.ppf(0.90)

print(f"Exponential model: X ~ Exponential(mean={mean_wait} seconds)")
print(f"(a) P(wait <= 5 s) = {prob_wait_le_5:.4f}")
print(f"(b) P(wait > 20 s) = {prob_wait_gt_20:.4f}")
print(f"(c) 90th percentile = {p90_wait:.3f} seconds")

t = np.linspace(0, 50, 400)
pdf_vals = queue_dist.pdf(t)
cdf_vals = queue_dist.cdf(t)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(t, pdf_vals, linewidth=2, label='PDF')
ax1.axvline(5, linestyle=':', alpha=0.7, label='5 seconds')
ax1.axvline(20, linestyle='--', alpha=0.7, label='20 seconds')
ax1.set_xlabel('Wait time (seconds)', fontsize=12)
ax1.set_ylabel('Probability density', fontsize=12)
ax1.set_title('Exponential PDF', fontsize=13, fontweight='bold')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=10)

ax2.plot(t, cdf_vals, linewidth=2, label='CDF')
ax2.axhline(0.90, linestyle=':', alpha=0.7, label='90th percentile')
ax2.axvline(p90_wait, linestyle=':', alpha=0.7)
ax2.set_xlabel('Wait time (seconds)', fontsize=12)
ax2.set_ylabel('Cumulative probability', fontsize=12)
ax2.set_title('Exponential CDF', fontsize=13, fontweight='bold')
ax2.set_ylim(0, 1.05)
ax2.grid(True, alpha=0.3)
ax2.legend(fontsize=10)

plt.tight_layout()
plt.show()

---

# Section 3: The Law of Large Numbers

**Lecture Connection:** In Lecture 7, you proved the Law of Large Numbers (LLN): as sample size grows, the sample mean converges to the true population mean. Now we'll *watch it happen* by simulation.

### Why LLN Matters in CS

- **Load testing:** Why does running 100 requests give reliable averages?
- **Monitoring:** Why do long-term average metrics stabilize?
- **Reliability:** Why can we trust empirical measurements over time?

**The Law of Large Numbers (informal):** For any ε > 0,
$$\lim_{n \to \infty} P\left(\left|\overline{X}_n - \mu\right| > \epsilon\right) = 0$$

In words: The sample mean converges to the true population mean as we increase sample size.

## 3.1: Demonstrating LLN with Different Distributions

In [ ]:
# Simulate LLN: as we collect more samples, sample mean → population mean

# Define population parameters
distributions = {
    'Normal(μ=100, σ=15)': stats.norm(loc=100, scale=15),
    'Exponential(mean=50)': stats.expon(scale=50),
    'Poisson(λ=20)': stats.poisson(mu=20),
}

# For each distribution, draw increasing sample sizes and track sample mean
sample_sizes = np.logspace(1, 4, 50).astype(int)  # 10, 20, ..., 10000

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (dist_name, dist) in zip(axes, distributions.items()):
    true_mean = dist.mean()
    sample_means = []
    
    for n in sample_sizes:
        sample = dist.rvs(size=n, random_state=rng)
        sample_means.append(np.mean(sample))
    
    # Plot
    ax.plot(sample_sizes, sample_means, 'o-', alpha=0.7, markersize=4, label='Sample mean')
    ax.axhline(true_mean, color='red', linestyle='--', linewidth=2, label=f'True mean = {true_mean:.2f}')
    ax.set_xscale('log')
    ax.set_xlabel('Sample Size (n)', fontsize=11)
    ax.set_ylabel('Sample Mean', fontsize=11)
    ax.set_title(f'LLN: {dist_name}', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Notice: As sample size increases (log scale), sample mean converges to true population mean.")


## 3.2: CS Application — Load Testing Convergence

In [ ]:
# Real-world scenario: Load testing a web service
# Individual response times are noisy, but averages stabilize

# Simulate response times (log-normal distribution, realistic for servers)
true_mean_response = 150  # milliseconds
mu_ln = np.log(true_mean_response) - 0.5 * 0.4**2  # adjust for mean
sigma_ln = 0.4
response_dist = stats.lognorm(s=sigma_ln, scale=np.exp(mu_ln))

# Run increasing test batch sizes
test_batch_sizes = np.array([5, 10, 20, 50, 100, 200, 500, 1000, 2000, 5000])
batch_averages = []
batch_stds = []

for batch_size in test_batch_sizes:
    responses = response_dist.rvs(size=batch_size, random_state=rng)
    batch_averages.append(np.mean(responses))
    batch_stds.append(np.std(responses))

# Plot
fig, ax = plt.subplots(figsize=(10, 6))

# Mean ± 1 std band
batch_means = np.array(batch_averages)
batch_std = np.array(batch_stds)
ax.fill_between(test_batch_sizes, batch_means - batch_std, batch_means + batch_std, 
                  alpha=0.3, color='blue', label='Mean ± 1 std')

# Sample means
ax.plot(test_batch_sizes, batch_means, 'o-', linewidth=2, markersize=8, 
        color='blue', label='Sample mean')

# True population mean
ax.axhline(response_dist.mean(), color='red', linestyle='--', linewidth=2, 
           label=f'True population mean = {response_dist.mean():.2f}ms')

ax.set_xscale('log')
ax.set_xlabel('Test Batch Size (requests)', fontsize=12)
ax.set_ylabel('Average Response Time (ms)', fontsize=12)
ax.set_title('Load Testing: Convergence of Average Response Time', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\nLoad Testing Convergence:")
print("Small batches: high variability, unreliable averages")
print("Large batches: low variability, stable averages")
print("This is why load tests need sufficient data volume!")


## 3.3: Exercise 7 — Law of Large Numbers

In [ ]:
# EXERCISE 7: Convergence of Sample Mean
# Create a simulation demonstrating LLN for Uniform(0, 100) distribution.
# Plot sample mean vs. sample size for n = 10, 20, 50, 100, 500, 1000, 5000.
# Show the true population mean (50) as a horizontal line.
# Comment: At what sample size does convergence "look good"?

rng_ex7 = np.random.default_rng(7)

uniform_dist = stats.uniform(loc=0, scale=100)
true_mean = uniform_dist.mean()
sample_sizes = np.array([10, 20, 50, 100, 500, 1000, 5000])

sample_means = []
for n in sample_sizes:
    sample = uniform_dist.rvs(size=n, random_state=rng_ex7)
    sample_means.append(np.mean(sample))

sample_means = np.array(sample_means)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sample_sizes, sample_means, 'o-', linewidth=2, markersize=8, label='Sample mean')
ax.axhline(true_mean, color='red', linestyle='--', linewidth=2, label=f'True mean = {true_mean:.0f}')
ax.set_xscale('log')
ax.set_xlabel('Sample size n (log scale)', fontsize=12)
ax.set_ylabel('Sample mean', fontsize=12)
ax.set_title('LLN for Uniform(0, 100)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

for n, mean_n in zip(sample_sizes, sample_means):
    print(f"n = {n:4d}: sample mean = {mean_n:.3f}, error = {mean_n - true_mean:+.3f}")

stable_idx = np.where(np.abs(sample_means - true_mean) < 2)[0]
if len(stable_idx) > 0:
    first_stable_n = sample_sizes[stable_idx[0]]
    print(f"\nConvergence starts to look good around n ≈ {first_stable_n}, and is clearly stable by n = 500 or 1000.")
else:
    print("\nBy visual inspection, convergence looks good once n reaches a few hundred samples.")

## 3.4: Exercise 8 — LLN Application

In [ ]:
# EXERCISE 8: API Error Rate Estimation
# Your API has a true error rate of 2%. You run tests with increasing batch sizes.
# Simulate error counts (Binomial) and plot the estimated error rate vs. batch size.
# At what batch size does the estimate stabilize (within ±0.5% of true rate)?

rng_ex8 = np.random.default_rng(8)

true_error_rate = 0.02
batch_sizes = np.array([10, 20, 50, 100, 200, 500, 1000, 2000, 5000, 10000])

# Generate one long stream of request outcomes, then compute cumulative estimates
max_n = batch_sizes.max()
errors = rng_ex8.binomial(n=1, p=true_error_rate, size=max_n)
cumulative_errors = np.cumsum(errors)
estimated_rates = cumulative_errors[batch_sizes - 1] / batch_sizes

tolerance = 0.005  # ±0.5%
within_tolerance = np.abs(estimated_rates - true_error_rate) <= tolerance

stable_n = None
for i, ok in enumerate(within_tolerance):
    if ok and np.all(within_tolerance[i:]):
        stable_n = batch_sizes[i]
        break

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(batch_sizes, estimated_rates, 'o-', linewidth=2, markersize=7, label='Estimated error rate')
ax.axhline(true_error_rate, color='red', linestyle='--', linewidth=2, label='True error rate = 2%')
ax.axhline(true_error_rate + tolerance, color='gray', linestyle=':', alpha=0.8, label='±0.5% band')
ax.axhline(true_error_rate - tolerance, color='gray', linestyle=':', alpha=0.8)
ax.set_xscale('log')
ax.set_xlabel('Batch size', fontsize=12)
ax.set_ylabel('Estimated error rate', fontsize=12)
ax.set_title('Estimating API Error Rate with Larger Batches', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

for n, rate in zip(batch_sizes, estimated_rates):
    print(f"n = {n:5d}: estimated error rate = {rate:.4%}")

if stable_n is not None:
    print(f"\nThe estimate stays within ±0.5% of the true rate from about n = {stable_n} onward in this simulation.")
else:
    print("\nIn this run the estimate does not remain inside the ±0.5% band until the largest batch size;")
    print("but it clearly becomes much more stable once n reaches the high hundreds to low thousands.")

---

# Section 4: The Central Limit Theorem

**Lecture Connection:** In Lecture 7, you proved the Central Limit Theorem (CLT): the distribution of sample means becomes approximately normal for large enough sample sizes, *regardless of the original distribution*. This is profound!

### Why CLT Matters in CS

- **Setting SLAs:** Even if response times are skewed, daily averages are approximately normal
- **Alerting:** We can compute confidence intervals and detect anomalies
- **Statistics:** Enables hypothesis testing and inference

**The Central Limit Theorem (informal):** If X₁, X₂, ..., Xₙ are i.i.d. from any distribution with mean μ and variance σ², then for large n:
$$\overline{X}_n \approx N\left(\mu, \frac{\sigma^2}{n}\right)$$

In words: The sample mean is approximately normal with mean μ and standard error σ/√n.

**Resources:**
- [Central Limit Theorem](https://en.wikipedia.org/wiki/Central_limit_theorem)

## 4.1: CLT Simulation — Exponential to Normal

In [ ]:
# Simulate CLT: draw repeated samples from exponential (non-normal),
# compute sample means for different sample sizes, and plot distributions

# Exponential distribution: clearly NOT normal (right-skewed)
lambda_rate = 0.02  # mean = 50
source_dist = stats.expon(scale=1/lambda_rate)
true_mean = source_dist.mean()
true_std = source_dist.std()

# Number of repeated experiments
n_experiments = 10000

# Different sample sizes to show CLT progression
sample_sizes = [5, 30, 100]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Row 1: Original distribution and samples
ax = axes[0, 0]
x_vals = np.linspace(0, 250, 1000)
ax.plot(x_vals, source_dist.pdf(x_vals), 'b-', linewidth=2, label='Original PDF (Exponential)')
ax.set_xlabel('Value', fontsize=11)
ax.set_ylabel('Probability Density', fontsize=11)
ax.set_title('Original Distribution (Non-Normal!)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Draw samples for visualization
sample_viz = source_dist.rvs(size=5000, random_state=rng)
ax = axes[0, 1]
ax.hist(sample_viz, bins=50, density=True, alpha=0.7, color='blue', edgecolor='black')
ax.plot(x_vals, source_dist.pdf(x_vals), 'r-', linewidth=2, label='True PDF')
ax.set_xlabel('Value', fontsize=11)
ax.set_ylabel('Frequency (density)', fontsize=11)
ax.set_title('Sample Data from Exponential', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Row 2: Distribution of sample means for different n
for idx, n in enumerate(sample_sizes):
    ax = axes[1, idx]
    
    # Generate n_experiments sample means, each from n observations
    sample_means = []
    for _ in range(n_experiments):
        sample = source_dist.rvs(size=n, random_state=rng)
        sample_means.append(np.mean(sample))
    
    sample_means = np.array(sample_means)
    
    # Plot histogram
    ax.hist(sample_means, bins=50, density=True, alpha=0.7, color='orange', edgecolor='black')
    
    # Overlay theoretical normal distribution
    theoretical_std = true_std / np.sqrt(n)
    normal_dist_theory = stats.norm(loc=true_mean, scale=theoretical_std)
    x_theory = np.linspace(sample_means.min(), sample_means.max(), 200)
    ax.plot(x_theory, normal_dist_theory.pdf(x_theory), 'r-', linewidth=2, 
            label=f'N({true_mean:.1f}, {theoretical_std:.2f}²)')
    
    ax.set_xlabel('Sample Mean', fontsize=11)
    ax.set_ylabel('Frequency (density)', fontsize=11)
    ax.set_title(f'Distribution of Sample Means (n={n})', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("Notice: The distribution of sample means becomes more normal as n increases!")
print(f"Despite the original being exponential (skewed), the sample means form a bell curve.")


## 4.2: Verifying Standard Error Formula: $σ/\sqrt{n}$

In [ ]:
# Verify that standard error of sample mean = σ / √n

print("Standard Error Verification:")
print(f"Population std (σ): {true_std:.4f}")
print()

for n in sample_sizes:
    # Theoretical standard error
    se_theoretical = true_std / np.sqrt(n)
    
    # Empirical standard error (from simulation)
    sample_means = []
    for _ in range(5000):
        sample = source_dist.rvs(size=n, random_state=rng)
        sample_means.append(np.mean(sample))
    se_empirical = np.std(sample_means, ddof=1)
    
    print(f"n = {n:3d}: Theoretical SE = {se_theoretical:.4f}, Empirical SE = {se_empirical:.4f}")


## 4.3: CLT with Different Source Distributions

In [ ]:
# Show CLT works for different source distributions
source_dists = {
    'Uniform(0, 100)': stats.uniform(loc=0, scale=100),
    'Binomial(n=20, p=0.3)': stats.binom(n=20, p=0.3),
    'Exponential(λ=0.02)': stats.expon(scale=50),
}

fig, axes = plt.subplots(2, 3, figsize=(16, 8))

# Fixed sample size to show CLT convergence
n_sample = 30
n_expts = 5000

for col, (dist_name, dist) in enumerate(source_dists.items()):
    # Top row: original distribution
    ax = axes[0, col]
    
    if 'Uniform' in dist_name or 'Exponential' in dist_name:
        x = np.linspace(dist.ppf(0.001), dist.ppf(0.999), 200)
        ax.plot(x, dist.pdf(x), 'b-', linewidth=2)
    else:  # Binomial
        x = np.arange(0, dist.ppf(0.999) + 1)
        ax.bar(x, dist.pmf(x), alpha=0.7, color='blue', edgecolor='black')
    
    ax.set_title(f'{dist_name} (Original)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Probability', fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Bottom row: distribution of sample means
    ax = axes[1, col]
    
    true_mean = dist.mean()
    true_std = dist.std()
    
    sample_means = []
    for _ in range(n_expts):
        if 'Binomial' in dist_name:
            sample = dist.rvs(size=n_sample, random_state=rng)
        else:
            sample = dist.rvs(size=n_sample, random_state=rng)
        sample_means.append(np.mean(sample))
    
    sample_means = np.array(sample_means)
    
    # Plot histogram
    ax.hist(sample_means, bins=40, density=True, alpha=0.7, color='orange', edgecolor='black')
    
    # Overlay normal
    se = true_std / np.sqrt(n_sample)
    x_norm = np.linspace(sample_means.min(), sample_means.max(), 200)
    norm_curve = stats.norm.pdf(x_norm, loc=true_mean, scale=se)
    ax.plot(x_norm, norm_curve, 'r-', linewidth=2, label='Normal approximation')
    
    ax.set_title(f'Sample Means (n={n_sample})', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sample Mean', fontsize=10)
    ax.set_ylabel('Frequency (density)', fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Central Limit Theorem: Different Source Distributions', fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()


## 4.4: CS Application — Daily Average Response Times

In [ ]:
# Real-world scenario: Individual request times are log-normal (skewed),
# but daily averages are approximately normal.
# This lets us set alerts for unusual daily performance.

# Individual response times: log-normal (like a real server)
mu_ln = 3.5
sigma_ln = 0.8
response_dist = stats.lognorm(s=sigma_ln, scale=np.exp(mu_ln))

# Simulate multiple days
requests_per_day = 100000
n_days = 1000

daily_averages = []
for day in range(n_days):
    daily_responses = response_dist.rvs(size=requests_per_day, random_state=rng)
    daily_averages.append(np.mean(daily_responses))

daily_averages = np.array(daily_averages)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Individual response times (highly skewed)
sample_responses = response_dist.rvs(size=100000, random_state=rng)
ax1.hist(sample_responses, bins=100, density=True, alpha=0.7, color='blue', edgecolor='black')
ax1.set_xlabel('Individual Response Time (ms)', fontsize=12)
ax1.set_ylabel('Frequency (density)', fontsize=12)
ax1.set_title('Individual Request Times (Log-Normal, Skewed)', fontsize=13, fontweight='bold')
ax1.set_xlim([0, 300])
ax1.grid(True, alpha=0.3, axis='y')

# Daily averages (approximately normal!)
ax2.hist(daily_averages, bins=50, density=True, alpha=0.7, color='orange', edgecolor='black', 
         label='Observed daily averages')

# Fit normal to daily averages
mean_daily = np.mean(daily_averages)
std_daily = np.std(daily_averages)
x_normal = np.linspace(daily_averages.min(), daily_averages.max(), 200)
normal_fit = stats.norm.pdf(x_normal, loc=mean_daily, scale=std_daily)
ax2.plot(x_normal, normal_fit, 'r-', linewidth=2, label=f'Normal({mean_daily:.2f}, {std_daily:.3f}²)')

# Mark ±2σ bands (95% confidence)
ax2.axvline(mean_daily - 2*std_daily, color='green', linestyle=':', linewidth=2, 
            label='Mean ± 2σ (95% CI)')
ax2.axvline(mean_daily + 2*std_daily, color='green', linestyle=':', linewidth=2)
ax2.axvline(mean_daily, color='red', linestyle='--', linewidth=1, alpha=0.5)

ax2.set_xlabel('Daily Average Response Time (ms)', fontsize=12)
ax2.set_ylabel('Frequency (density)', fontsize=12)
ax2.set_title(f'Daily Averages (CLT: Approximately Normal)', fontsize=13, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Daily Average Response Time Statistics:")
print(f"  Mean: {mean_daily:.3f} ms")
print(f"  Std: {std_daily:.3f} ms")
print(f"  95% of days fall in range: [{mean_daily - 2*std_daily:.3f}, {mean_daily + 2*std_daily:.3f}] ms")
print(f"\nApplication: Set alerts if daily average > {mean_daily + 2*std_daily:.3f} ms")


## 4.5: Connection to Confidence Intervals (Preview)

The CLT enables **confidence intervals** — a key tool for inference.

For large n, by CLT:
$$\overline{X}_n \approx N\left(\mu, \frac{\sigma^2}{n}\right)$$

So a **95% confidence interval** for μ is:
$$\overline{X} \pm 1.96 \cdot \frac{\sigma}{\sqrt{n}}$$

*(We'll dive deeper into confidence intervals in the next tutorial.)*

In [ ]:
# Quick preview: confidence intervals from CLT
# Simulate unknown population, estimate μ with confidence interval

# Unknown population (we only have a sample)
true_population = stats.expon(scale=100)
true_mean_pop = true_population.mean()
true_std_pop = true_population.std()

# Draw ONE sample
sample = true_population.rvs(size=500, random_state=rng)
sample_mean = np.mean(sample)
sample_std = np.std(sample, ddof=1)  # unbiased estimate

# By CLT, standard error ≈ sample_std / √n
n = len(sample)
se = sample_std / np.sqrt(n)

# 95% confidence interval (z = 1.96 for 95%)
z_95 = stats.norm.ppf(0.975)  # two-tailed
ci_lower = sample_mean - z_95 * se
ci_upper = sample_mean + z_95 * se

print(f"Sample size: {n}")
print(f"Sample mean: {sample_mean:.3f}")
print(f"Sample std (unbiased): {sample_std:.3f}")
print(f"Standard error: {se:.3f}")
print()
print(f"95% Confidence Interval for μ: [{ci_lower:.3f}, {ci_upper:.3f}]")
print(f"True population mean: {true_mean_pop:.3f}")
print(f"CI contains true mean? {ci_lower <= true_mean_pop <= ci_upper}")


## 4.6: Exercise 9 — Central Limit Theorem

In [ ]:
# EXERCISE 9: CLT from Uniform Distribution
# Source distribution: Uniform(0, 10)
# Take 5000 repeated samples of size n, compute sample means.
# Plot histogram of sample means and overlay the theoretical normal distribution.
# Try for n = 10, 30, 100. Comment on how well the normal approximation fits.

rng_ex9 = np.random.default_rng(9)

source_dist = stats.uniform(loc=0, scale=10)
true_mean = source_dist.mean()
true_std = source_dist.std()

n_experiments = 5000
sample_sizes = [10, 30, 100]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, n in zip(axes, sample_sizes):
    samples = source_dist.rvs(size=(n_experiments, n), random_state=rng_ex9)
    sample_means = samples.mean(axis=1)

    ax.hist(sample_means, bins=40, density=True, alpha=0.7, edgecolor='black',
            label='Simulated sample means')

    se = true_std / np.sqrt(n)
    x = np.linspace(sample_means.min(), sample_means.max(), 400)
    ax.plot(x, stats.norm(loc=true_mean, scale=se).pdf(x), linewidth=2,
            label='Theoretical normal')

    ax.set_title(f'n = {n}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Sample mean', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f"Source distribution mean = {true_mean:.2f}, std = {true_std:.4f}")
print("The normal approximation is already decent at n=10, good at n=30, and very close by n=100.")

## 4.7: Exercise 10 — Standard Error and Sample Size

In [ ]:
# EXERCISE 10: Effect of Sample Size on Standard Error
# For Normal(μ=50, σ=10), compute sample means for n = 5, 10, 20, 50, 100.
# For each n, run 1000 experiments and compute the empirical standard error.
# Plot empirical SE vs. theoretical SE = σ/√n.
# How does doubling n affect standard error?

rng_ex10 = np.random.default_rng(10)

mu = 50
sigma = 10
normal_dist = stats.norm(loc=mu, scale=sigma)

sample_sizes = np.array([5, 10, 20, 50, 100])
n_experiments = 1000

empirical_se = []
theoretical_se = sigma / np.sqrt(sample_sizes)

for n in sample_sizes:
    samples = normal_dist.rvs(size=(n_experiments, n), random_state=rng_ex10)
    sample_means = samples.mean(axis=1)
    empirical_se.append(np.std(sample_means, ddof=1))

empirical_se = np.array(empirical_se)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(sample_sizes, empirical_se, 'o-', linewidth=2, markersize=8, label='Empirical SE')
ax.plot(sample_sizes, theoretical_se, 's--', linewidth=2, markersize=7, label='Theoretical SE = σ/√n')
ax.set_xlabel('Sample size n', fontsize=12)
ax.set_ylabel('Standard error', fontsize=12)
ax.set_title('Standard Error Shrinks as Sample Size Grows', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

for n, emp, theo in zip(sample_sizes, empirical_se, theoretical_se):
    print(f"n = {n:3d}: empirical SE = {emp:.4f}, theoretical SE = {theo:.4f}")

print("\nDoubling n reduces the standard error by a factor of 1/sqrt(2) ≈ 0.707,")
print("so the uncertainty shrinks by about 29%, not by half.")

## 4.8: Exercise 11 — Confidence Intervals

In [ ]:
# EXERCISE 11: Confidence Interval from Sample
# Generate a sample of size n=100 from Normal(μ=200, σ=30).
# Compute the 95% confidence interval for μ using the CLT formula.
# Repeat 100 times and count how many intervals contain the true μ=200.
# You should get approximately 95 intervals.

rng_ex11 = np.random.default_rng(11)

true_mu = 200
true_sigma = 30
n = 100
n_repeats = 100
z_95 = stats.norm.ppf(0.975)

ci_bounds = []
covers_true_mu = []

for _ in range(n_repeats):
    sample = rng_ex11.normal(loc=true_mu, scale=true_sigma, size=n)
    sample_mean = np.mean(sample)
    sample_std = np.std(sample, ddof=1)
    se = sample_std / np.sqrt(n)

    ci_lower = sample_mean - z_95 * se
    ci_upper = sample_mean + z_95 * se

    ci_bounds.append((ci_lower, ci_upper, sample_mean))
    covers_true_mu.append(ci_lower <= true_mu <= ci_upper)

ci_bounds = np.array(ci_bounds)
covers_true_mu = np.array(covers_true_mu)
n_cover = covers_true_mu.sum()

print(f"Intervals containing the true mean ({true_mu}): {n_cover} out of {n_repeats}")
print(f"Coverage proportion = {n_cover / n_repeats:.2%}")

fig, ax = plt.subplots(figsize=(11, 8))
for i, (lower, upper, mean_hat) in enumerate(ci_bounds[:30], start=1):
    color = 'tab:blue' if covers_true_mu[i-1] else 'tab:red'
    ax.plot([lower, upper], [i, i], color=color, linewidth=2)
    ax.plot(mean_hat, i, 'o', color=color, markersize=4)

ax.axvline(true_mu, color='black', linestyle='--', linewidth=2, label='True mean')
ax.set_xlabel('Confidence interval for μ', fontsize=12)
ax.set_ylabel('Replication (first 30 shown)', fontsize=12)
ax.set_title('95% Confidence Intervals Across Repeated Samples', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---

## Summary & Key Takeaways

### What You've Learned

1. **Discrete Distributions:**
   - Bernoulli, Binomial, Poisson model counts of events
   - Use PMF for exact probabilities, CDF for cumulative probabilities
   - CS: model error counts, request successes, rare events

2. **Continuous Distributions:**
   - Normal, Exponential model measurements in time/space
   - Use PDF for density, CDF for cumulative probabilities, PPF for quantiles
   - CS: model response times, session durations, waiting times

3. **Law of Large Numbers:**
   - Sample mean converges to true population mean as n grows
   - Why load testing works, why metrics stabilize over time

4. **Central Limit Theorem:**
   - Sample means are approximately normal for large n, regardless of source distribution
   - Enables confidence intervals and hypothesis testing
   - Standard error = σ/√n: larger samples → tighter estimates


### Key Formulas

| Concept | Formula | Meaning |
|---------|---------|----------|
| Binomial PMF | $P(X=k) = C(n,k) p^k (1-p)^{(n-k)}$ | Probability of k successes in n trials |
| Poisson PMF | $P(X=k) = \frac{(λ^k e^{-λ})} {k!}$ | Probability of k events in interval |
| Standard Error | $SE = σ / \sqrt{n}$ | Std of sample mean |
| CLT | $\overline{X}_n \sim N(μ, σ²/n)$ | Sample mean is approximately normal |
| 95% CI | $\overline{X} \pm 1.96 \cdot SE$ | 95% confidence interval for μ |

### Code Patterns to Remember

```python
# Create a distribution
dist = stats.binom(n=10, p=0.5)  # or norm, expon, poisson, ...

# Probability calculations
dist.pmf(k)   # discrete: P(X = k)
dist.pdf(x)   # continuous: f(x)
dist.cdf(x)   # P(X <= x)
dist.ppf(p)   # inverse CDF: quantile
dist.rvs(size=n)  # random samples

# Parameters
dist.mean(), dist.var(), dist.std()
```